# Deriving Logical Operators of a Stabilizer Code

**Input:** a stabilizer code as a list of Pauli strings (its generators).
**Output:** the code's logical operators, as Pauli strings.

A Pauli operator is a *logical operator* when it
1. commutes with every stabilizer (undetectable — no syndrome fires), and
2. is not itself a product of stabilizers (so it acts non-trivially on encoded states).

The tool turns both conditions into linear algebra over GF(2) (arithmetic mod 2).

## 1. Encoding: Pauli strings ↔ binary vectors

Each Pauli on `n` qubits becomes a `2n`-bit vector `(x-part | z-part)`:
`I=(0|0)`, `X=(1|0)`, `Z=(0|1)`, `Y=(1|1)`.

In [4]:
def pauli_to_binary(p):
    """Pauli string -> 2n-bit list [x-part | z-part]."""
    x = [1 if c in "XY" else 0 for c in p]   # x-bit set where there's an X or a Y
    z = [1 if c in "ZY" else 0 for c in p]   # z-bit set where there's a Z or a Y
    return x + z

def binary_to_pauli(v):
    """2n-bit list [x-part | z-part] -> Pauli string (inverse of pauli_to_binary)."""
    n = len(v) // 2
    letters = []
    for i in range(n):
        x_bit, z_bit = v[i], v[i + n]        # qubit i's two bits, n apart
        if   x_bit == 0 and z_bit == 0: letters.append("I")
        elif x_bit == 1 and z_bit == 0: letters.append("X")
        elif x_bit == 0 and z_bit == 1: letters.append("Z")
        else:                           letters.append("Y")   # (1,1)
    return "".join(letters)

## 2. Mod-2 linear algebra toolkit

General-purpose GF(2) machinery — no quantum content. Row reduction over the
field {0, 1}, where addition is XOR (1+1 = 0). `row_reduce` puts a bit-matrix
into reduced echelon form and reports its pivot positions.

In [6]:
def add_rows(r1, r2):
    """XOR two bit-rows together (addition mod 2)."""
    return [(a + b) % 2 for a, b in zip(r1, r2)]

def find_pivot_row(M, col, start_row):
    """First row at or below start_row with a 1 in this column, else None."""
    for r in range(start_row, len(M)):
        if M[r][col] == 1:
            return r
    return None

def row_reduce(M):
    """Reduce M to echelon form over GF(2).
    Returns (reduced matrix, list of (row, col) pivot positions)."""
    M = [row[:] for row in M]          # work on a copy, leave the original alone
    n_rows, n_cols = len(M), len(M[0])
    pivot_row, pivots = 0, []
    for col in range(n_cols):                      # sweep left to right
        pr = find_pivot_row(M, col, pivot_row)
        if pr is None:
            continue                               # no pivot -> free column
        M[pivot_row], M[pr] = M[pr], M[pivot_row]  # bring pivot row into place
        for r in range(n_rows):                    # clear this column elsewhere
            if r != pivot_row and M[r][col] == 1:
                M[r] = add_rows(M[r], M[pivot_row])
        pivots.append((pivot_row, col))
        pivot_row += 1
    return M, pivots

## 3. The two conditions as linear algebra

### Condition 1 — commute with every stabilizer

Commutation uses the *symplectic* product `a·d + b·c` (the halves cross over).
`swap_halves` swaps a stabilizer's x- and z-parts so a plain dot product computes
it: then "commutes with all stabilizers" becomes "lies in the null space".

In [8]:
def swap_halves(row):
    """Swap the x-part and z-part of a row, so a plain dot product
    against it computes the symplectic (commutation) product."""
    n = len(row) // 2
    return row[n:] + row[:n]          # z-part first, x-part second

### Solving condition 1 → a basis for N(S)

Reduce the swapped check matrix; non-pivot columns are *free variables*.
Setting each free variable to 1 in turn (others 0) and solving the pivots
yields one basis vector per free column — together they span N(S),
every operator commuting with all stabilizers.

In [10]:
def null_space_basis(reduced, pivots, n_cols):
    """Basis for the null space of a reduced GF(2) matrix.
    One basis vector per free (non-pivot) column."""
    pivot_cols = [c for (r, c) in pivots]
    free_cols  = [c for c in range(n_cols) if c not in pivot_cols]
    basis = []
    for f in free_cols:
        v = [0] * n_cols
        v[f] = 1                         # turn this free variable on, others off
        for (r, c) in pivots:            # solve each pivot from that choice
            v[c] = reduced[r][f]
        basis.append(v)
    return basis

### Condition 2 — not a product of stabilizers

`reduce_vector` cancels a vector against a set of (reduced) rows by XOR-ing them in.
If a candidate from N(S) reduces to all-zeros it *was* a product of stabilizers
(discard); a non-zero remainder means it points outside S — a genuine logical.

In [12]:
def reduce_vector(v, basis_rows):
    """Reduce v against a set of rows by XOR-ing in any whose leading 1
    matches a 1 in v. Returns the remainder (all-zeros => v was in their span)."""
    v = v[:]                                  # copy; don't mutate the caller's vector
    for row in basis_rows:
        pivot = next((i for i, b in enumerate(row) if b == 1), None)   # row's leading 1
        if pivot is not None and v[pivot] == 1:
            v = add_rows(v, row)              # XOR it away
    return v

## 4. The driver

`extract_logicals` applies condition 2: from a basis for N(S), keep one
representative per coset of S (discarding stabilizers and de-duplicating
logicals that differ only by a stabilizer). `derive_logicals` chains the
whole pipeline: Pauli strings in, logical operators out.

In [14]:
def extract_logicals(reduced_stab, nspace_basis):
    """Condition 2: keep N(S) vectors that survive reduction against the
    stabilizers and against logicals already accepted (one per coset of S)."""
    logicals = []
    for v in nspace_basis:
        remainder = reduce_vector(v, reduced_stab + logicals)
        if any(bit == 1 for bit in remainder):
            logicals.append(remainder)        # keep the reduced survivor
    return logicals

def derive_logicals(stabilizers):
    """Logical operators of a stabilizer code, as Pauli strings.
    Input: list of stabilizer generators as Pauli strings."""
    H = [pauli_to_binary(s) for s in stabilizers]
    n_cols = len(H[0])

    # condition 1: basis for N(S) (everything commuting with all stabilizers)
    reduced, pivots = row_reduce([swap_halves(row) for row in H])
    nspace = null_space_basis(reduced, pivots, n_cols)

    # condition 2: strip out the stabilizers, keep one rep per logical
    reduced_stab, _ = row_reduce(H)
    logicals = extract_logicals(reduced_stab, nspace)

    return [binary_to_pauli(v) for v in logicals]

## 5. Validation

Two independent checks on every result: the number of logicals equals 2k
(for k logical qubits), and every returned operator genuinely commutes with
all stabilizers. The commutation check uses the symplectic product directly,
not the pipeline's machinery, so it's an independent second opinion.

In [16]:
def symplectic(a, b):
    """Symplectic product of two binary vectors, mod 2 (0 = commute, 1 = anticommute)."""
    n = len(a) // 2
    return sum(a[i]*b[i+n] + a[i+n]*b[i] for i in range(n)) % 2

def commutes_with_all(pauli, stabilizers):
    """Does this Pauli string commute with every stabilizer?"""
    v = pauli_to_binary(pauli)
    return all(symplectic(v, pauli_to_binary(s)) == 0 for s in stabilizers)

def check(name, stabilizers):
    """Run derive_logicals and report both sanity checks."""
    logs = derive_logicals(stabilizers)
    n, k = len(stabilizers[0]), len(stabilizers[0]) - len(stabilizers)
    count_ok = (len(logs) == 2 * k)
    comm_ok  = all(commutes_with_all(p, stabilizers) for p in logs)
    print(f"{name}: {logs}")
    print(f"   count == 2k ({2*k}): {count_ok}   |   all commute with S: {comm_ok}")

In [17]:
check("bit-flip   ", ["ZZI", "IZZ"])
check("phase-flip ", ["XXI", "IXX"])
check("Steane     ", ["IIIXXXX","IXXIIXX","XIXIXIX","IIIZZZZ","IZZIIZZ","ZIZIZIZ"])
check("5-qubit    ", ["XZZXI","IXZZX","XIXZZ","ZXIXZ"])
check("Shor       ", ["ZZIIIIIII","IZZIIIIII","IIIZZIIII","IIIIZZIII",
                      "IIIIIIZZI","IIIIIIIZZ","XXXXXXIII","IIIXXXXXX"])

bit-flip   : ['XXX', 'IIZ']
   count == 2k (2): True   |   all commute with S: True
phase-flip : ['IIX', 'ZZZ']
   count == 2k (2): True   |   all commute with S: True
Steane     : ['IIXIXXI', 'IIZIZZI']
   count == 2k (2): True   |   all commute with S: True
5-qubit    : ['ZIIZX', 'ZZZZZ']
   count == 2k (2): True   |   all commute with S: True
Shor       : ['IIIIIIXXX', 'IIZIIZIIZ']
   count == 2k (2): True   |   all commute with S: True


In [27]:
check("4-1-2 surface", ["XXXX", "ZIZI", "IZIZ"])

4-1-2 surface: ['IXIX', 'IIZZ']
   count == 2k (2): True   |   all commute with S: True
